# Credit Card Fraud Detection using an Artificial Neural Network

This recruiter-facing notebook documents the analytical workflow. Reusable
logic is maintained in `src/` so the same preprocessing and inference contract
is used by training, evaluation, testing, and the Streamlit application.

## Business question

Can an ANN identify rare fraudulent transactions while controlling both missed
fraud and false customer alerts?

Because only 0.1727% of the source transactions are fraudulent, accuracy alone
is not an adequate success measure. The analysis emphasizes precision, recall,
F1-score, ROC-AUC, PR-AUC, the confusion matrix, and decision-threshold trade-offs.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FEATURE_COLUMNS
from src.data_preprocessing import load_dataset, split_dataset, fit_and_transform
from src.model_training import balanced_class_weights, build_ann, train

## 1. Load and validate the dataset

Download the public `creditcard.csv` file as described in
`data/README_data.md`, then place it under `data/`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"
fraud_df = load_dataset(DATA_PATH)

print(f"Shape: {fraud_df.shape}")
print(f"Missing values: {int(fraud_df.isna().sum().sum())}")
print(f"Fraud transactions: {int(fraud_df['Class'].sum()):,}")
print(f"Fraud rate: {fraud_df['Class'].mean():.4%}")
fraud_df.head()

## 2. Inspect class imbalance

In [ ]:
class_counts = (
    fraud_df["Class"]
    .map({0: "Legitimate", 1: "Fraud"})
    .value_counts()
)
display(class_counts.to_frame("transactions"))

ax = class_counts.plot(kind="bar", figsize=(7, 4), title="Class Distribution")
ax.set_xlabel("")
ax.set_ylabel("Transactions")
plt.tight_layout()
plt.show()

A majority-class-only model would appear highly accurate but would detect no
fraud. The minority class therefore requires explicit training and evaluation
attention.

## 3. Create stratified data splits

In [ ]:
splits = split_dataset(fraud_df)
scaler, X_train, X_valid, X_test = fit_and_transform(splits)

split_summary = pd.DataFrame(
    {
        "rows": [
            len(splits.X_train),
            len(splits.X_valid),
            len(splits.X_test),
        ],
        "fraud_rows": [
            int(splits.y_train.sum()),
            int(splits.y_valid.sum()),
            int(splits.y_test.sum()),
        ],
        "fraud_rate": [
            splits.y_train.mean(),
            splits.y_valid.mean(),
            splits.y_test.mean(),
        ],
    },
    index=["Train", "Validation", "Test"],
)
split_summary

The scaler is fitted only on the training data. Validation and test features are
transformed using those training statistics, which prevents preprocessing
leakage.

## 4. Calculate imbalance-aware class weights

In [ ]:
class_weights = balanced_class_weights(splits.y_train.to_numpy())
class_weights

Class weights increase the contribution of rare fraud rows to the training
loss. The default pipeline uses class weighting rather than SMOTE because the
input features are already anonymized principal components, and synthetic
interpolation may not represent plausible transactions.

## 5. Review the ANN architecture

In [ ]:
model = build_ann(
    input_dim=len(FEATURE_COLUMNS),
    hidden_units=64,
    dropout_rate=0.20,
    learning_rate=0.001,
)
model.summary()

The improved training model uses ReLU hidden layers, batch normalization,
dropout, a sigmoid output, binary cross-entropy, and Adam. It tracks both ROC
and precision-recall AUC during training.

## 6. Train and evaluate

In [ ]:
# Set to True when the full dataset is available and you want to retrain.
RUN_TRAINING = False

if RUN_TRAINING:
    training_result = train(
        dataset_path=DATA_PATH,
        params_path=PROJECT_ROOT / "models" / "credit_card_best_params.json",
        epochs=40,
        threshold_strategy="max_f1",
    )
    display(pd.Series(training_result["test_metrics"]))
else:
    print(
        "Training skipped. Set RUN_TRAINING = True to generate a new "
        "class-weighted model and evaluation outputs."
    )

## 7. Inspect the audited supplied-model results

In [ ]:
metrics_path = PROJECT_ROOT / "outputs" / "model_metrics.json"
reference = json.loads(metrics_path.read_text())
reference_metrics = reference["reference_test_metrics_at_threshold_0_5"]

pd.Series(
    {
        "Accuracy": reference_metrics["accuracy"],
        "Precision": reference_metrics["precision"],
        "Recall": reference_metrics["recall"],
        "F1-score": reference_metrics["f1_score"],
        "ROC-AUC": reference_metrics["roc_auc"],
        "PR-AUC": reference_metrics["pr_auc"],
    }
).to_frame("value")

In [ ]:
figure_paths = [
    PROJECT_ROOT / "outputs" / "confusion_matrix.png",
    PROJECT_ROOT / "outputs" / "roc_curve.png",
    PROJECT_ROOT / "outputs" / "precision_recall_curve.png",
]

for path in figure_paths:
    image = plt.imread(path)
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(path.stem.replace("_", " ").title())
    plt.show()

## 8. Deployment

The inference pipeline validates the exact 30-feature contract, applies the
saved scaler, produces fraud probabilities, and converts them to operational
labels at a configurable threshold.

Run the web application from the project root:

```bash
streamlit run app/streamlit_app.py
```

See `README_HOSTING.md` for Streamlit Community Cloud deployment.

## Key takeaways

- Accuracy is misleading for this 0.1727%-fraud dataset.
- PR-AUC directly reflects ranking performance on the rare fraud class.
- Class weights address imbalance without changing the original observations.
- The operating threshold should be selected using validation data and aligned
  with the business cost of false positives versus false negatives.
- A deployable portfolio project needs consistent preprocessing, saved schemas,
  reusable inference code, documentation, and responsible-use limitations.